In [1]:
import pandas as pd
import plotly.graph_objects as go

master = pd.read_csv('../data/clean/master.csv')
before, after = '2017-11', '2017-12'

In [2]:
def month_stats(m):
    d = master[master['order_month'] == m]
    orders = d['order_id'].nunique()
    revenue = d['price'].sum()
    return orders, revenue, revenue / orders

o0, r0, a0 = month_stats(before)
o1, r1, a1 = month_stats(after)

volume_effect = (o1 - o0) * a0
value_effect  = (a1 - a0) * o1

print('Revenue change:', round(r1 - r0))
print('Volume effect (fewer orders):', round(volume_effect))
print('Value effect (smaller baskets):', round(value_effect))
print('Check (should be 0):', round((r1 - r0) - (volume_effect + value_effect)))

Revenue change: -261615
Volume effect (fewer orders): -240543
Value effect (smaller baskets): -21072
Check (should be 0): 0


In [3]:
cat = master.pivot_table(index='category_en', columns='order_month',
                         values='price', aggfunc='sum', fill_value=0)
cat['change'] = cat[after] - cat[before]
cat['change'].sort_values().head(10)

category_en
bed_bath_table               -37876.47
computers_accessories        -32247.65
furniture_decor              -31018.18
watches_gifts                -25735.42
cool_stuff                   -18181.09
health_beauty                -17585.65
garden_tools                 -17479.49
telephony                    -13337.19
office_furniture             -11234.83
agro_industry_and_commerce    -8640.79
Name: change, dtype: float64

In [4]:
top = cat['change'].sort_values().head(5)
other = cat['change'].sum() - top.sum()

labels = [before] + list(top.index) + ['all other', after]
values = [r0] + list(top.values) + [other, 0]
measure = ['absolute'] + ['relative'] * (len(top) + 1) + ['total']

fig = go.Figure(go.Waterfall(x=labels, y=values, measure=measure))
fig.update_layout(title=f'Revenue bridge: {before} to {after}')
fig.show()

Revenue fell 26.5% from Nov to Dec 2017 (987,648 to 726,033 BRL). Fewer orders explain about 92% of the drop and smaller baskets about 8%. Five categories account for 55% of the loss, and new customers account for about 99% of the lost buyers, so the drop reflects fewer first-time buyers after Black Friday rather than a retention problem.